# Supply Chain Performance & Risk Analytics
## Phase 1 — Exploratory Data Analysis & Cleaning
**Dataset:** USAID SCMS Delivery History Dataset  
**Rows:** 10,324 | **Columns:** 33  
**Author:** Manohar Choudhary

In [1]:
# import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
print(os.getcwd())


C:\Users\lenovo\OneDrive\Desktop\supply-chain-performance-analytics\python_code


In [21]:
df = pd.read_csv(
    "../data/raw_data/SCMS_Delivery_History_Dataset_20150929.csv",
    encoding="latin1"
)
print(df.shape)

(10324, 33)


In [29]:
df.shape

(10324, 33)

In [3]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (10324, 33)
Columns: ['ID', 'Project Code', 'PQ #', 'PO / SO #', 'ASN/DN #', 'Country', 'Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'PQ First Sent to Client Date', 'PO Sent to Vendor Date', 'Scheduled Delivery Date', 'Delivered to Client Date', 'Delivery Recorded Date', 'Product Group', 'Sub Classification', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Dosage Form', 'Unit of Measure (Per Pack)', 'Line Item Quantity', 'Line Item Value', 'Pack Price', 'Unit Price', 'Manufacturing Site', 'First Line Designation', 'Weight (Kilograms)', 'Freight Cost (USD)', 'Line Item Insurance (USD)']


## Initial Exploration
Checking data types, null counts, and basic statistics before cleaning.

In [4]:
# datatype and null inspection

print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nNull percentages:")
print((df.isnull().sum() / len(df)) * 100)

ID                                int64
Project Code                     object
PQ #                             object
PO / SO #                        object
ASN/DN #                         object
Country                          object
Managed By                       object
Fulfill Via                      object
Vendor INCO Term                 object
Shipment Mode                    object
PQ First Sent to Client Date     object
PO Sent to Vendor Date           object
Scheduled Delivery Date          object
Delivered to Client Date         object
Delivery Recorded Date           object
Product Group                    object
Sub Classification               object
Vendor                           object
Item Description                 object
Molecule/Test Type               object
Brand                            object
Dosage                           object
Dosage Form                      object
Unit of Measure (Per Pack)        int64
Line Item Quantity                int64


In [17]:
#Column investigation (Freight Cost, Weight, Shipment Mode)

print("Freight Cost sample values:")
print(df["Freight Cost (USD)"].tail(10))

print("\nWeight sample values:")
print(df["Weight (Kilograms)"].tail(10))

print("\nShipment Mode unique values:")
print(df["Shipment Mode"].value_counts(dropna=False))

Freight Cost sample values:
10314        NaN
10315    26180.0
10316     3410.0
10317        NaN
10318        NaN
10319        NaN
10320        NaN
10321        NaN
10322        NaN
10323        NaN
Name: Freight Cost (USD), dtype: float64

Weight sample values:
10314        NaN
10315    15198.0
10316     1547.0
10317        NaN
10318        NaN
10319        NaN
10320        NaN
10321        NaN
10322     1392.0
10323        NaN
Name: Weight (Kilograms), dtype: float64

Shipment Mode unique values:
Shipment Mode
Air            6113
Truck          2830
Air Charter     650
Ocean           371
NaN             360
Name: count, dtype: int64


## Data Cleaning
All cleaning steps applied in sequence. Each step documented in issue_log.md

In [24]:
# Issue 006 - Freight Type flag
df["Freight_Type"] = np.where(
    df["Freight Cost (USD)"].str.contains(r"[A-Za-z]", na=False),
    "Absorbed", "Separate"
)

# Issue 006 - Convert Freight Cost to numeric
df["Freight Cost (USD)"] = pd.to_numeric(
    df["Freight Cost (USD)"], errors="coerce"
)

# Issue 007 - Weight Type flag
df["Weight_Type"] = np.where(
    df["Weight (Kilograms)"].str.contains(r"[A-Za-z]", na=False),
    "Captured Separately", "Recorded"
)

# Issue 007 - Convert Weight to numeric
df["Weight (Kilograms)"] = pd.to_numeric(
    df["Weight (Kilograms)"], errors="coerce"
)

# Issue 005 - Convert date columns
# ── Issue 005 ── Convert three clean date columns
date_columns = [
    "Scheduled Delivery Date",
    "Delivered to Client Date",
    "Delivery Recorded Date"
]
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="%d-%b-%y")

# ── Issue 009 ── Convert two mixed date columns
mixed_date_columns = [
    "PQ First Sent to Client Date",
    "PO Sent to Vendor Date"
]
for col in mixed_date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="%m/%d/%Y")

# Issue 004 - Fill insurance nulls
df["Line Item Insurance (USD)"] = df["Line Item Insurance (USD)"].fillna(0)

print("Cleaning complete.")
print("Shape after cleaning:", df.shape)

Cleaning complete.
Shape after cleaning: (10324, 35)


In [25]:
# varification after cleaning

print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nData types after cleaning:")
print(df.dtypes)

print("\nFreight_Type distribution:")
print(df["Freight_Type"].value_counts())

print("\nWeight_Type distribution:")
print(df["Weight_Type"].value_counts())

Remaining nulls:
Shipment Mode                    360
PQ First Sent to Client Date    2681
PO Sent to Vendor Date          5732
Dosage                          1736
Weight (Kilograms)              3952
Freight Cost (USD)              4126
dtype: int64

Data types after cleaning:
ID                                       int64
Project Code                            object
PQ #                                    object
PO / SO #                               object
ASN/DN #                                object
Country                                 object
Managed By                              object
Fulfill Via                             object
Vendor INCO Term                        object
Shipment Mode                           object
PQ First Sent to Client Date    datetime64[ns]
PO Sent to Vendor Date          datetime64[ns]
Scheduled Delivery Date         datetime64[ns]
Delivered to Client Date        datetime64[ns]
Delivery Recorded Date          datetime64[ns]
Product Group  

In [6]:
# Flag whether Dosage is applicable for the product group
df["Dosage_Applicable"] = np.where(
    df["Product Group"].isin(["HRDT", "MRDT"]),
    "Not Applicable", "Applicable"
)

# Verify
print(df["Dosage_Applicable"].value_counts())
print(df[["Product Group", "Dosage", "Dosage_Applicable"]].head(20))

Dosage_Applicable
Applicable        8588
Not Applicable    1736
Name: count, dtype: int64
   Product Group     Dosage Dosage_Applicable
0           HRDT        NaN    Not Applicable
1            ARV    10mg/ml        Applicable
2           HRDT        NaN    Not Applicable
3            ARV      150mg        Applicable
4            ARV       30mg        Applicable
5            ARV    10mg/ml        Applicable
6            ARV      200mg        Applicable
7            ARV      200mg        Applicable
8            ARV       30mg        Applicable
9            ARV   200/50mg        Applicable
10           ARV   200/50mg        Applicable
11          HRDT        NaN    Not Applicable
12          HRDT        NaN    Not Applicable
13           ARV  150/300mg        Applicable
14          HRDT        NaN    Not Applicable
15           ARV      200mg        Applicable
16          HRDT        NaN    Not Applicable
17          HRDT        NaN    Not Applicable
18           ARV         2g        A

In [65]:
df.shape


(10324, 36)

# Feature engineering -

**Creating new column for analysis assistance**

In [30]:
# ── Feature Engineering ──
df["Delivery_Delay_Days"] = (
    df["Delivered to Client Date"] - df["Scheduled Delivery Date"]
).dt.days

df["On_Time_Delivery"] = np.where(
    df["Delivery_Delay_Days"] <= 2, "On Time", "Late"
)

df["Delivery_Status"] = np.where(
    df["Delivery_Delay_Days"] < 0, "Early",
    np.where(df["Delivery_Delay_Days"] <= 2, "On Time", "Late")
)

# ── Export cleaned data ──
df.to_csv(
    "../data/processed_data/scms_cleaned.csv",
    index=False,
    encoding="utf-8"
)
print("Exported. Final shape:", df.shape)
print("Columns:", df.columns.tolist())

Exported. Final shape: (10324, 38)
Columns: ['ID', 'Project Code', 'PQ #', 'PO / SO #', 'ASN/DN #', 'Country', 'Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'PQ First Sent to Client Date', 'PO Sent to Vendor Date', 'Scheduled Delivery Date', 'Delivered to Client Date', 'Delivery Recorded Date', 'Product Group', 'Sub Classification', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Dosage Form', 'Unit of Measure (Per Pack)', 'Line Item Quantity', 'Line Item Value', 'Pack Price', 'Unit Price', 'Manufacturing Site', 'First Line Designation', 'Weight (Kilograms)', 'Freight Cost (USD)', 'Line Item Insurance (USD)', 'Freight_Type', 'Weight_Type', 'Delivery_Delay_Days', 'On_Time_Delivery', 'Delivery_Status']


In [14]:
df[["Delivery_Delay_Days","On_Time_Delivery"]]

,Delivery_Delay_Days,On_Time_Delivery
0,0,on-time
1,0,on-time
2,0,on-time
3,0,on-time
4,0,on-time
...,...,...
10319,-16,on-time
10320,6,late
10321,-6,on-time
10322,-36,on-time


In [16]:
df[["Delivery_Delay_Days","On_Time_Delivery","delivery_status"]]

,Delivery_Delay_Days,On_Time_Delivery,delivery_status
0,0,on-time,on-time
1,0,on-time,on-time
2,0,on-time,on-time
3,0,on-time,on-time
4,0,on-time,on-time
...,...,...,...
10319,-16,on-time,Early
10320,6,late,Late
10321,-6,on-time,Early
10322,-36,on-time,Early


In [19]:
df[["Delivery_Delay_Days","On_Time_Delivery","Delivery_Status"]]

,Delivery_Delay_Days,On_Time_Delivery,Delivery_Status
0,0,On Time,On Time
1,0,On Time,On Time
2,0,On Time,On Time
3,0,On Time,On Time
4,0,On Time,On Time
...,...,...,...
10319,-16,On Time,Early
10320,6,Late,Late
10321,-6,On Time,Early
10322,-36,On Time,Early
